# 02 Temporal Patterns

Build hourly, daily, and monthly temporal demand profiles and export them for the Streamlit Temporal Patterns page.

Exports:
- temporal_patterns_hourly.csv
- temporal_patterns_daily.csv
- temporal_patterns_monthly.csv

In [1]:
import sys
import importlib
import pandas as pd

sys.path.insert(0, '..')
import utils as _utils
importlib.reload(_utils)

load_app_ready = _utils.load_app_ready
export_df = _utils.export_df

In [2]:
df = load_app_ready()
df.shape

(15907082, 22)

In [3]:
required = ['city_name', 'year', 'start_hour', 'day_of_week', 'day_name', 'month', 'month_name', 'trip_id']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns for temporal analysis: {missing}')

base = df.dropna(subset=['city_name', 'year']).copy()
base['year'] = pd.to_numeric(base['year'], errors='coerce')
base = base.dropna(subset=['year'])
base['year'] = base['year'].astype(int)

hourly = (
    base.dropna(subset=['start_hour', 'day_of_week', 'day_name'])
    .groupby(['city_name', 'year', 'day_of_week', 'day_name', 'start_hour'], as_index=False)
    .agg(trips=('trip_id', 'count'))
    .sort_values(['city_name', 'year', 'day_of_week', 'start_hour'])
)

daily = (
    base.dropna(subset=['day_of_week', 'day_name'])
    .groupby(['city_name', 'year', 'day_of_week', 'day_name'], as_index=False)
    .agg(trips=('trip_id', 'count'))
    .sort_values(['city_name', 'year', 'day_of_week'])
)

monthly = (
    base.dropna(subset=['month', 'month_name'])
    .groupby(['city_name', 'year', 'month', 'month_name'], as_index=False)
    .agg(trips=('trip_id', 'count'))
    .sort_values(['city_name', 'year', 'month'])
)

hourly.head(), daily.head(), monthly.head()

(  city_name  year  day_of_week day_name  start_hour  trips
 0    Bergen  2018            0   Monday           4    147
 1    Bergen  2018            0   Monday           5    812
 2    Bergen  2018            0   Monday           6   1305
 3    Bergen  2018            0   Monday           7    931
 4    Bergen  2018            0   Monday           8    652,
   city_name  year  day_of_week   day_name  trips
 0    Bergen  2018            0     Monday  16918
 1    Bergen  2018            1    Tuesday  17686
 2    Bergen  2018            2  Wednesday  16480
 3    Bergen  2018            3   Thursday  16268
 4    Bergen  2018            4     Friday  15850,
   city_name  year  month month_name  trips
 0    Bergen  2018      6       June    642
 1    Bergen  2018      7       July  19823
 2    Bergen  2018      8     August  23919
 3    Bergen  2018      9  September  18618
 4    Bergen  2018     10    October  17131)

In [4]:
export_df('temporal_patterns_hourly', hourly)
export_df('temporal_patterns_daily', daily)
export_df('temporal_patterns_monthly', monthly)
print('Exported temporal pattern tables.')

[bridge] Exported DataFrame: temporal_patterns_hourly.csv (target: NOTEBOOK_EXPORTS_PATH)
[bridge] Exported DataFrame: temporal_patterns_daily.csv (target: NOTEBOOK_EXPORTS_PATH)
[bridge] Exported DataFrame: temporal_patterns_monthly.csv (target: NOTEBOOK_EXPORTS_PATH)
Exported temporal pattern tables.
